# Rank Ensemble (Final Submission)

Combines predictions from TF-IDF, LSTM, and DeBERTa models using a **weighted rank ensemble**. Each model assigns a score to each option based on its rank in the top-3, then scores are summed across models with the strongest model (DeBERTa) getting the highest weight.

This is the final submission notebook — it takes individual model CSVs and produces a single blended prediction file.

### 1. Experiment Tracking (WandB)

In [1]:
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    wandb.init(
        project='23f2003236-t22026',
        name='ensemble',
        config={
            'tfidf_weight': 0.18,
            'lstm_weight': 0.27,
            'deberta_weight': 0.55,
            'method': 'rank_ensemble',
            'models': ['tfidf', 'lstm', 'deberta']
        }
    )
    print('WandB connected')
except Exception:
    print('WandB not available, continuing offline')
    USE_WANDB = False

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f2003236 (23f2003236-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260711_171245-1sbeawqa
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ensemble
wandb: ⭐️ View project at https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026
wandb: 🚀 View run at https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026/runs/1sbeawqa


WandB connected


### 2. Load Individual Model Submissions

Each CSV has `ID` and `Prediction` columns where `Prediction` is a space-separated string of 3 answer letters (e.g. `"B D A"`).

In [2]:
import pandas as pd
import numpy as np
from collections import defaultdict

# file paths to individual model submissions
path_tfidf  = '/kaggle/input/datasets/lukerohan/mcq-solver-csv/submission_tfidf.csv'
path_lstm   = '/kaggle/input/datasets/lukerohan/mcq-solver-csv/submission_lstm.csv'
path_deberta = '/kaggle/input/datasets/lukerohan/mcq-solver-csv/submission_DeBERTa.csv'

sub_tfidf  = pd.read_csv(path_tfidf)
sub_lstm   = pd.read_csv(path_lstm)
sub_deberta = pd.read_csv(path_deberta)

print(f'TF-IDF:  {sub_tfidf.shape}')
print(f'LSTM:    {sub_lstm.shape}')
print(f'DeBERTa: {sub_deberta.shape}')

# verify the ame number of rows
assert len(sub_tfidf) == len(sub_lstm) == len(sub_deberta), 'Row count mismatch!'

TF-IDF:  (500, 2)
LSTM:    (500, 2)
DeBERTa: (500, 2)


### 3. Weighted Rank Ensemble

Each model's top-3 prediction contributes a weighted score based on rank:

- Rank 1 gets `weight * 1.0`
- Rank 2 gets `weight * 0.5`
- Rank 3 gets `weight * 0.333`

Scores are summed across all 3 models per option letter, then the top-3 options by total score become the final prediction.

In [3]:
# ensemble weights (strongest model gets highest weight)
W_TFIDF  = 0.18
W_LSTM   = 0.27
W_DEBERTA = 0.55

final_predictions = []

for idx in range(len(sub_tfidf)):
    scores = defaultdict(float)

    # score TF-IDF predictions
    preds = str(sub_tfidf.iloc[idx]['Prediction']).split()
    for rank, pred in enumerate(preds):
        scores[pred] += W_TFIDF * (1.0 / (rank + 1))

    # score LSTM predictions
    preds = str(sub_lstm.iloc[idx]['Prediction']).split()
    for rank, pred in enumerate(preds):
        scores[pred] += W_LSTM * (1.0 / (rank + 1))

    # score DeBERTa predictions
    preds = str(sub_deberta.iloc[idx]['Prediction']).split()
    for rank, pred in enumerate(preds):
        scores[pred] += W_DEBERTA * (1.0 / (rank + 1))

    # take top-3 options by combined score
    best_3 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]
    final_predictions.append(' '.join([x[0] for x in best_3]))

print(f'Ensemble complete. {len(final_predictions)} predictions generated.')

Ensemble complete. 500 predictions generated.


### 4. Save & Log

In [4]:
# build submission dataframe
id_col = sub_tfidf.columns[0]
final_submission = pd.DataFrame({
    id_col: sub_tfidf[id_col],
    'Prediction': final_predictions
})

In [5]:
# save
save_path = '/kaggle/working/ensemble.csv'
final_submission.to_csv(save_path, index=False)

# log artifact to WandB
try:
    artifact = wandb.Artifact(
        name='final-ensemble-submission',
        type='submission'
    )
    artifact.add_file(save_path)
    wandb.log_artifact(artifact)
except Exception:
    pass

print(f'Saved: {save_path}')
print(final_submission.head())

try:
    wandb.finish()
except Exception:
    pass

wandb: updating run metadata; uploading artifact final-ensemble-submission


Saved: /kaggle/working/ensemble.csv
   ID Prediction
0   1      A E B
1   2      B D A
2   3      B E A
3   4      E C D
4   5      C A D


wandb: 🚀 View run ensemble at: https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026/runs/1sbeawqa
wandb: ⭐️ View project at: https://wandb.ai/23f2003236-iit-madras/23f2003236-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260711_171245-1sbeawqa/logs
wandb: WARNING Artifact "final-ensemble-submission" already exists with the same content. No new version will be created.
